# Post-prediction evaluation
This notebook starts from `test_predictions.csv`, reproduces the bootstrap analysis, and applies Holm correction to the bootstrap p-values.

In [ ]:
!pip install -q pandas numpy scikit-learn krippendorff statsmodels

In [ ]:
import os

import numpy as np
import pandas as pd
import krippendorff
from sklearn.metrics import accuracy_score, f1_score
from statsmodels.stats.multitest import multipletests

INPUT_PATH = '/content/drive/MyDrive/Models/evaluation/test_predictions.csv'
OUTPUT_DIR = '/content/drive/MyDrive/Models/evaluation'
LANGUAGES = ['EN', 'IT', 'SI']
N_BOOT = 5000
RANDOM_STATE = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

test_predictions = pd.read_csv(INPUT_PATH)
if 'lang' not in test_predictions.columns and 'language' in test_predictions.columns:
    test_predictions = test_predictions.rename(columns={'language': 'lang'})

required_columns = {'lang', 'label', 'pred_human', 'pred_llm'}
missing_columns = required_columns.difference(test_predictions.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

test_df = test_predictions.copy()
print(f'Loaded {len(test_df)} rows from {INPUT_PATH}')
display(test_df.head())

In [ ]:
def macro_f1(y, pred):
    return f1_score(y, pred, average='macro')

def accuracy(y, pred):
    return accuracy_score(y, pred)

def alpha_metric(y, pred):
    return krippendorff.alpha(
        reliability_data=[y.tolist(), pred.tolist()],
        level_of_measurement='ordinal'
    )

METRICS = {
    'Macro F1': macro_f1,
    'Accuracy': accuracy,
    'Alpha': alpha_metric,
}

def paired_bootstrap(y_true, pred_human, pred_llm, metric_fn, n_boot=N_BOOT, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    n = len(y_true)

    observed = metric_fn(y_true, pred_human) - metric_fn(y_true, pred_llm)
    diffs = np.empty(n_boot)

    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        ph = pred_human[idx]
        pl = pred_llm[idx]
        diffs[b] = metric_fn(yt, ph) - metric_fn(yt, pl)

    return {
        'observed': observed,
        'median': np.median(diffs),
        'ci_low': np.percentile(diffs, 2.5),
        'ci_high': np.percentile(diffs, 97.5),
        'distribution': diffs,
    }

def bootstrap_p_value(diffs):
    lower_tail = np.mean(diffs <= 0)
    upper_tail = np.mean(diffs >= 0)
    return min(1.0, 2.0 * min(lower_tail, upper_tail))

In [ ]:
bootstrap_rows = []

for lang in LANGUAGES + ['ALL']:
    subset = test_df if lang == 'ALL' else test_df[test_df['lang'] == lang]

    y_true = subset['label'].to_numpy()
    pred_human = subset['pred_human'].to_numpy()
    pred_llm = subset['pred_llm'].to_numpy()

    for metric_name, metric_fn in METRICS.items():
        result = paired_bootstrap(
            y_true,
            pred_human,
            pred_llm,
            metric_fn,
            n_boot=N_BOOT,
            random_state=RANDOM_STATE
        )

        bootstrap_rows.append({
            'language': lang,
            'metric': metric_name,
            'observed': result['observed'],
            'median': result['median'],
            'ci_low': result['ci_low'],
            'ci_high': result['ci_high'],
            'p_value': bootstrap_p_value(result['distribution']),
        })

bootstrap_df = pd.DataFrame(bootstrap_rows)
display(bootstrap_df)
bootstrap_df.to_csv(os.path.join(OUTPUT_DIR, 'bootstrap_results.csv'), index=False)

In [ ]:
pvals = bootstrap_df['p_value'].to_numpy()
reject_holm, pvals_holm, _, _ = multipletests(pvals, method='holm')

holm_results = bootstrap_df.copy()
holm_results['p_value_holm'] = pvals_holm
holm_results['reject_holm'] = reject_holm

display(holm_results)
holm_results.to_csv(os.path.join(OUTPUT_DIR, 'holm_results.csv'), index=False)